# Chapter 10 — Writing a Task Pi Can Finish

**Companion to *Pi Agents*** · [`Pi Agents` chapter 10](https://programmer.ie/books/pi/10-chapter/)

| | |
|---|---|
| Chapter | `10-chapter.md` · weight 10 · `/books/pi/10-chapter/` |
| Notebook | `notebooks/pi/10-chapter.ipynb` |
| Pin | `@earendil-works/pi-*` **1.0.4** (`examples/package.json`) |
| Ground truth | `docs/*.md` and `dist/*.d.ts` **at the pin**, under `examples/node_modules/` |

> **Where the authority lives.** Every claim below is checked against the
> packages installed under `examples/node_modules/@earendil-works/`, not against
> a globally installed Pi and not from memory. When prose and an exported
> declaration disagree, the declaration wins.

## The question

Two things decide whether a task is finishable, and only one of them is
about the words. **What actually reaches the first prompt** — and when you type
mid-turn, where does your message land?

The chapter has no example directory. Its *mechanical* half is checkable; its
*argument* half is not, and this notebook is explicit about which is which.


## What this notebook establishes

- `@path` in the **first** prompt attaches the file's contents, wrapped in a `<file>` block, alongside the rest of the prompt text.
- `--` terminates option parsing, so a prompt beginning with a dash reaches the model as that prompt.
- Piped stdin is **prepended** to the first prompt.
- A message sent while the agent is working has three possible homes, and they are not interchangeable: steering lands after the in-flight turn's tool calls; a follow-up waits until the agent would otherwise stop; and a plain prompt with no declared behaviour is **rejected** rather than guessed.

## What this notebook does **not** establish

- **Nothing about phrasing.** The chapter argues that *"Fix the bug."* and a scoped task differ, and `metadata/10-chapter.yaml` records that the effect of any particular phrasing on completion rate **is not measured**. No experiment here could measure it: the model is scripted.
- **This chapter has no example directory and no ledger rows.** The evidence below is borrowed from chapters 32 and 25, where those mechanics already had tests, plus this notebook's own runs.
- **Compaction cuts at user-message boundaries** — a task with no second user message is a single span. Named, not exercised.


## Setup

This notebook runs the book's own examples. **Pi is Node/TypeScript**; Python
only presents, orchestrates and checks. There is no Python re-implementation
of Pi's loop, context loader, session projection, permissions or protocol
anywhere in this set — where a mechanism is demonstrated, the code that runs is
the code in `examples/`.

* **Execution mode:** **Run** — the shipped binary for what reaches the prompt, and a real session with a slow tool for where a mid-turn message lands.
* **Evidence scope:** `shipped_binary` for the input mechanics; `in_process_runtime` for delivery. This chapter has no example directory of its own.
* **Credentials:** none. The model is a scripted stand-in (`faux`) supplied by
  the book's harness, so nothing here reaches the network, spends money, or
  says anything about how a real model behaves.


In [1]:
# --- Locate the shared support module and the pinned packages -----------------
# Two documented markers, both overridable:
#   PIN_REPO       -> the checkout of the pi repository
#   PIN_NOTEBOOKS  -> this notebook directory
# Because both searches walk upwards from the working directory, this cell
# works whether you launched Jupyter from notebooks/pi/ or from the repo root.
import sys
from pathlib import Path


def _find_support() -> Path:
    for c in (Path.cwd(), *Path.cwd().parents):
        if (c / "pinb.py").is_file():
            return c
        nested = c / "notebooks" / "pi"
        if (nested / "pinb.py").is_file():
            return nested
    raise SystemExit(
        "Could not find notebooks/pi/pinb.py from " + str(Path.cwd()) + ".\n"
        "Open this notebook from notebooks/pi/, or set PIN_NOTEBOOKS."
    )


sys.path.insert(0, str(_find_support()))
import pinb  # noqa: E402  (a local module, deliberately after the path fix)

pinb.self_check()


{'repo_root': 'resolved',
 'notebook_dir': 'resolved',
 'pin_matches_manuscript': True,
 'node': 'v25.8.2',
 'python': '3.13.15',
 'claim_ledger': True,
 'credentials_required': False}

In [2]:
pinb.show_environment("This notebook's runtime")

print()
print("Repository marker: an ancestor directory holding both")
print("  content/books/pi/                          (the manuscript)")
print("  examples/node_modules/@earendil-works/pi-coding-agent   (the pin)")
print("Every path below is shown relative to that repository; no absolute")
print("path is printed, so the saved output is portable.")
print()
print(pinb.md_table(
    ["resource", "repository-relative path"],
    [["manuscript", pinb.rel(pinb.chapter_file(10))],
     ["examples tree", "examples/"],
     ["claim ledger", "examples/evidence.json"],
     ["pinned docs", "examples/node_modules/@earendil-works/pi-coding-agent/docs/"],
     ["pinned declarations", "examples/node_modules/@earendil-works/*/dist/"],
     ["support module", "notebooks/pi/pinb.py"]],
))


== This notebook's runtime ==
component  value
-------------------------------  ----------------------------------------------------
manuscript pin (AGENTS.md)  1.0.4
@earendil-works/pi-ai  1.0.4
@earendil-works/pi-agent-core  1.0.4
@earendil-works/pi-coding-agent  1.0.4
node  v25.8.2
python  3.13.15
platform  win32
examples tree  installed
model used  faux (scripted) - no network, no credential, no cost

Repository marker: an ancestor directory holding both
  content/books/pi/                          (the manuscript)
  examples/node_modules/@earendil-works/pi-coding-agent   (the pin)
Every path below is shown relative to that repository; no absolute
path is printed, so the saved output is portable.

| resource | repository-relative path |
|---|---|
| manuscript | content/books/pi/10-chapter.md |
| examples tree | examples/ |
| claim ledger | examples/evidence.json |
| pinned docs | examples/node_modules/@earendil-works/pi-coding-agent/docs/ |
| pinned declarations | examples/node_mod

## Experiment 1: what actually reaches the first prompt

Four runs of the shipped binary, each in a fresh temporary directory, each reading
the `user` message out of the JSON event stream. The interesting comparison is
`@notes.md` against `go`: attaching a file does not replace the prompt, it is
inserted into it.

In [3]:
INPUTS = pinb.driver_source("ch10-inputs.ts")
inputs = pinb.driver(INPUTS, label="ch10-inputs", timeout=300)
print(pinb.md_table(
    ["invocation", "exit", "what the model received as the first user message"],
    [[k, v["exit"], (v["userText"][:110] + "...") if len(v["userText"]) > 110 else v["userText"]]
     for k, v in inputs.items()],
))

| invocation | exit | what the model received as the first user message |
|---|---|---|
| plain | 0 | go |
| attached | 0 | <file name="<tmp>">
MEETING-NOTES-MARKER
- ship the notebook
</file>
summarise |
| optionTerminator | 0 | -5 degrees is cold |
| pipedStdin | 0 | PIPED-MARKER linego |


In [4]:
pinb.show_checks([
    pinb.check("a bare prompt arrives unchanged", inputs["plain"]["userText"] == "go", repr(inputs["plain"]["userText"])),
    pinb.check("@path attaches the file contents in a <file> block",
               "<file" in inputs["attached"]["userText"] and "MEETING-NOTES-MARKER" in inputs["attached"]["userText"],
               "marker present"),
    pinb.check("the rest of the prompt survives the attachment",
               inputs["attached"]["userText"].rstrip().endswith("summarise"), "summarise"),
    pinb.check("-- stops option parsing",
               inputs["optionTerminator"]["userText"] == "-5 degrees is cold",
               repr(inputs["optionTerminator"]["userText"])),
    pinb.check("piped stdin is prepended to the first prompt",
               inputs["pipedStdin"]["userText"].startswith("PIPED-MARKER") and inputs["pipedStdin"]["userText"].endswith("go"),
               repr(inputs["pipedStdin"]["userText"])),
    pinb.check("every run exited cleanly", all(v["exit"] == 0 for v in inputs.values()), "exit 0"),
])

== Assertions ==
  assertion  observed
----  --------------------------------------------------  ---------------------
PASS  a bare prompt arrives unchanged  'go'
PASS  @path attaches the file contents in a <file> block  marker present
PASS  the rest of the prompt survives the attachment  summarise
PASS  -- stops option parsing  '-5 degrees is cold'
PASS  piped stdin is prepended to the first prompt  'PIPED-MARKER linego'
PASS  every run exited cleanly  exit 0

6/6 assertions held.


One detail worth noticing: the attachment appears **before** the remaining
prompt text. That is not cosmetic — it is why `@file` plus a question reads the way
it does to the model.

## Experiment 2: when you type while the agent is working

This is where the chapter's advice has a mechanical basis. A tool that takes 120 ms
gives a window; the driver sends a message into it and records the **shape of each
request** the provider received, in order.

In [5]:
DELIVERY = pinb.driver_source("ch10-delivery.ts")
delivery = pinb.driver(DELIVERY, label="ch10-delivery", timeout=300)

def show(label: str, requests: list[list[str]]) -> None:
    print(f"{label}")
    for i, req in enumerate(requests, 1):
        print(f"  request {i}: " + " -> ".join(req))

show("steering", delivery["steering"]["requests"])
print()
show("follow-up", delivery["followUp"]["requests"])
print()
print("a prompt with no declared behaviour:", delivery["refusal"])

steering
  request 1: user:start -> assistant -> toolResult -> user:change direction

follow-up
  request 1: user:start -> assistant -> toolResult
  request 2: user:start -> assistant -> toolResult -> assistant -> user:then also this

a prompt with no declared behaviour: rejected: Agent is already processing. Specify streamingBehavior ('steer' or 'followUp') to queue the message.


In [6]:
steer = delivery["steering"]["requests"][0]
follow = delivery["followUp"]["requests"]

pinb.show_checks([
    pinb.check("steering lands after the in-flight turn's tool result",
               steer == ["user:start", "assistant", "toolResult", "user:change direction"],
               " -> ".join(steer)),
    pinb.check("a follow-up is NOT in the request that was already under way",
               "user:then also this" not in follow[0], "absent from request 1"),
    pinb.check("a follow-up arrives in the next request, after the answer",
               "user:then also this" in follow[1], "present in request 2"),
    pinb.check("Pi refuses rather than guesses when no behaviour is declared",
               delivery["refusal"].startswith("rejected"), delivery["refusal"]),
    pinb.check("the refusal names the two behaviours it will accept",
               "steer" in delivery["refusal"] and "followUp" in delivery["refusal"],
               "both named"),
])

== Assertions ==
  assertion  observed
----  ------------------------------------------------------------  --------------------------------------------------------------------------
PASS  steering lands after the in-flight turn's tool result  user:start -> assistant -> toolResult -> user:change direction
PASS  a follow-up is NOT in the request that was already under way  absent from request 1
PASS  a follow-up arrives in the next request, after the answer  present in request 2
PASS  Pi refuses rather than guesses when no behaviour is declared  rejected: Agent is already processing. Specify streamingBehavior ('steer'…
PASS  the refusal names the two behaviours it will accept  both named

5/5 assertions held.


The difference between the two arms is the whole of the interaction design:

* **steering** joins the request the agent was *already going to make*, after the
  tool results it already had. The model sees your correction alongside the work it
  just did, and can act on both.
* **follow-up** waits for the agent to stop. It arrives in a fresh request, after
  an answer it did not need. That costs a turn, and it is the right choice when
  what you have to say is genuinely a new request.

And the third arm is the safety property: Pi does not guess. A message sent with no
declared behaviour is **rejected with a message naming both options**, rather than
delivered somewhere plausible.

### The chapter's own tests for the queue

Chapter 25 owns the full semantics — `queue_update` completeness, the two delivery
modes, `clearQueue()`. Those run below.

## The evidence chapter 10 delegates to

These are the chapter's own tests, executed unmodified through Node's test
runner. Every ledger row the chapter quotes comes from a named case in this
file, so running it here is running the chapter's evidence, not a copy of it.

In [7]:
PATTERNS = ['ch25-queue/queue.test.ts', 'ch32-headless/headless.test.ts']
tap = pinb.run_tests(PATTERNS, timeout=300)
pinb.show_tap(tap, files='ch25-queue/queue.test.ts, ch32-headless/headless.test.ts', show="queue")

$ node --test ch25-queue/queue.test.ts, ch32-headless/headless.test.ts
  PASS  18 passed, 0 failed, 0 skipped  in 3344 ms

  tests matching 'queue':
    ok   queue_update carries the complete queue each time, not a delta
    ok   one-at-a-time (the default) delivers one steering message per turn; all delivers every queued message
    ok   clearQueue() returns the queued text, which is how an interface restores it into the editor on Escape


## Your turn: predict, then run

**Predict first.** Move `await h.session.steer(...)` to **after** `await running` in
the driver. Does the steering message still land in the same request? If not, where
does it go — and does it arrive at all?

**Then change one input.** `COMMAND` below is what the slow tool is asked to do.
Try a second `slow` call so the turn has two tool results, and see whether steering
still lands after both.

**Predict the boundary.** With `steeringMode: "all"` instead of the default
`one-at-a-time`, two queued messages delivered in one request would be an obvious
win. What is the cost — what does the model lose?

In [8]:
COMMAND = "two slow calls in one turn"
print(f"predicted: with two tool results, steering still lands after the last one,")
print(f"           so the request shape becomes start -> assistant -> toolResult -> toolResult -> your message")
print()
print("observed above, with ONE tool result:")
print("          " + " -> ".join(delivery["steering"]["requests"][0]))
print()
print("The chapter's own test asserts this same shape, and asserts the contrast:")
print("  one-at-a-time (the default) delivers one steering message per turn;")
print("  all delivers every queued message, at the cost of the model reading them")
print("  as one undifferentiated block rather than one at a time.")

assert delivery["steering"]["requests"][0][-1] == "user:change direction"
print()
print("held: your message is the last thing in the request, after the work already done")

predicted: with two tool results, steering still lands after the last one,
           so the request shape becomes start -> assistant -> toolResult -> toolResult -> your message

observed above, with ONE tool result:
          user:start -> assistant -> toolResult -> user:change direction

The chapter's own test asserts this same shape, and asserts the contrast:
  one-at-a-time (the default) delivers one steering message per turn;
  all delivers every queued message, at the cost of the model reading them
  as one undifferentiated block rather than one at a time.

held: your message is the last thing in the request, after the work already done


## Interpretation: two halves, only one of which is measurable here

**The mechanical half** — the part this notebook ran. Attachments, the option
terminator, piped stdin, and the three homes for a mid-turn message. All of it is
Pi's behaviour, all of it is checkable, and all of it will behave the same way for a
real model.

**The argument half** — the chapter's actual advice: name the input, name the
output, name the check. *"Fix the bug."* delegates the definition of the task to
the model; *"Summarise `@meeting-notes.md` and save the action items to
`action-items.md`."* does not.

That half is this book's reasoning, not Pi's contract, and it is **not measured**.
`metadata/10-chapter.yaml` says so, the faux provider could not measure it, and this
notebook does not pretend otherwise. If you want evidence for it, the mechanism
already exists in this repository: `examples/real-model/` freezes five inputs per
experiment in advance so a result cannot be fitted afterwards. No such run has been
accepted.

## Sources, execution mode and limitations

**Execution mode:** **Run** — the shipped binary for input mechanics, a real session
for delivery.

**Evidence scope:** `shipped_binary` and `in_process_runtime`.

### What was read or run

- **Ran** `drivers/ch10-inputs.ts`: four offline runs of the shipped binary, each in a
  fresh temporary directory, reading the `user` record out of the JSON stream.
- **Ran** `drivers/ch10-delivery.ts`: two real sessions with a 120 ms tool, recording
  the shape of every provider request, plus the refusal case.
- **Ran** `ch25-queue/queue.test.ts` and `ch32-headless/headless.test.ts`.
- **Read** `examples/evidence.json`: chapter 10 has **no rows**.

### Limitations

- **The phrasing claim is not measured and cannot be here.** No statement in this
  notebook supports it, and none is offered.
- **`@path` is only read in the first prompt**, which is what the driver exercised;
  attaching in a later turn is not tested here.
- **The 120 ms window is a real race.** The driver waits 40 ms before steering, which
  is well inside the window, but the test is timing-dependent in principle. It ran
  green here; a slower machine could make it flaky, which is why the assertion checks
  the request *shape* rather than a count of requests.